# Repo and secured financing

**Prerequisites:** **05**. **`repo`** prices collateralized borrowing with **haircuts** and **repo rate** inputs.


## Concept

- **Collateral + haircut** govern lent securities vs cash.
- **Term vs overnight** changes accrual horizon and balance-sheet usage.
- **Implied financing** metrics summarize the carry embedded in the box.


In [ ]:
import json
from datetime import date

from finstack.valuations import ValuationResult, price_instrument_with_metrics, validate_instrument_json
from finstack.core.market_data import DiscountCurve, MarketContext

print("Imports OK (repo).")


## Instrument JSON


In [ ]:
AS_OF = date(2025, 1, 15)
AS_OF_STR = AS_OF.isoformat()

repo = json.dumps(
    {
        "type": "repo",
        "spec": {
            "id": "REPO-GC-7D",
            "repo_type": "Term",
            "start_date": AS_OF_STR,
            "maturity": "2025-01-22",
            "day_count": "Act360",
            "bdc": "following",
            "calendar_id": "usny",
            "repo_rate": "0.0525",
            "haircut": 0.02,
            "cash_amount": {"amount": "10000000", "currency": "USD"},
            "collateral": {
                "collateral_type": "General",
                "instrument_id": "UST-10Y",
                "market_value_id": "UST_10Y_PRICE",
                "quantity": 10000.0,
            },
            "discount_curve_id": "USD-OIS",
            "triparty": False,
            "attributes": {},
            "pricing_overrides": {},
        },
    }
)

try:
    validate_instrument_json(repo)
    print("repo JSON OK")
except Exception as exc:
    print("validate:", type(exc).__name__, exc)


## MarketContext


In [ ]:
ois = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (0.02, 0.999), (1.0, 0.97)],
    day_count="act_365f",
)
mc = MarketContext().insert(ois)
state = json.loads(mc.to_json())
state.setdefault("prices", {})
state["prices"]["UST_10Y_PRICE"] = {"price": {"amount": 98.5, "currency": "USD"}}
market_json = json.dumps(state)
print("collateral price id UST_10Y_PRICE injected")


## Pricing


In [ ]:
try:
    out = price_instrument_with_metrics(repo, market_json, AS_OF_STR, model="discounting")
    print(ValuationResult.from_json(out))
except Exception as exc:
    print("price:", type(exc).__name__, exc)


## Metrics


In [ ]:
try:
    out = price_instrument_with_metrics(
        repo,
        market_json,
        AS_OF_STR,
        model="discounting",
        metrics=["implied_financing_rate", "funding_cost", "collateral_haircut01"],
    )
    vr = ValuationResult.from_json(out)
    for m in ("implied_financing_rate", "funding_cost", "collateral_haircut01"):
        v = vr.get_metric(m)
        if v is not None:
            print(m, ":", v)
except Exception as exc:
    print("metrics:", type(exc).__name__, exc)


## Takeaways

- **Haircut** and **collateral marks** (`market_value_id`) drive security-for-cash economics.
- **Implied financing rate** links observed market levels to the repo structure.
- Overnight vs term is mostly a **schedule + rate** choice in JSON.
